# 04 – Seguimiento, Control y Alarmas

**Objetivo de este notebook**

Enriquecer el seguimiento del proyecto con métricas de forecasting y preparar la base para un sistema de alarmas de gestión.

**Bloque actual:** 2.1 – Enriquecer Project Tracking con forecasting

# Setup

In [1]:
from pathlib import Path
import openpyxl
import pandas as pd
import numpy as np
from datetime import datetime

BASE_DIR = Path.cwd()
DATA_ORIGINAL = BASE_DIR / "Data_Original"
EXCEL_FILE = DATA_ORIGINAL / "C2011-13 Wind Farm.xlsx"
OUTPUT_DIR = BASE_DIR / "04_Outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Cargamos también lo que ya teníamos de la fase anterior
PREV_OUTPUT = BASE_DIR / "03_Outputs"

print("Rutas configuradas")

# FASE 2.1 - Enriquecer Project Tracking con forecasting

## Analisis de Tracking Overview

In [2]:
wb = openpyxl.load_workbook(EXCEL_FILE, data_only=True)
ws = wb["Tracking Overview"]

print("Columnas disponibles en Tracking Overview (fila 2):")
print("-" * 60)

for col in range(1, 40):
    value = ws.cell(2, col).value
    if value is not None:
        print(f"Columna {col:2d}: {value}")

### Conclusiones Iniciales

Columnas más útiles que vamos a recuperar.

De todo ese listado, nos quedamos con las más relevantes para gestión:

- Indicadores básicos (ya los teníamos):
    - PV, EV, AC
    
    - SV, CV, SPI, CPI
    
    - SV(t), SPI(t), p-factor

- Forecasting de coste (EAC):

    - EAC (PF=1)

    - EAC (PF=CPI)

    - EAC (PF=SPI)

    - EAC (PF=SPI(t))

    - EAC (PF=0.8*CPI+0.2*SPI)

    - EAC (PF=0.8*CPI+0.2*SPI(t))

- Forecasting de plazo (EAC(t)):

    - EAC(t)-PV (PF=1)

    - EAC(t)-ES (PF=1)

    - EAC(t)-ES (PF=SPI(t))

## Extraer Project Tracking enriquecido

In [3]:
ws = wb["Tracking Overview"]

records = []
for row in range(3, ws.max_row + 1):
    tp_name = ws.cell(row, 1).value
    if tp_name is None:
        continue

    records.append({
        # Identificación
        "TP": int(tp_name) if str(tp_name).isdigit() else tp_name,
        "Start_Tracking_Period": ws.cell(row, 2).value,
        "Status_Date": ws.cell(row, 3).value,

        # Valores básicos
        "PV": ws.cell(row, 4).value,
        "EV": ws.cell(row, 5).value,
        "AC": ws.cell(row, 6).value,
        "ES": ws.cell(row, 7).value,

        # Variaciones e índices
        "SV": ws.cell(row, 8).value,
        "SPI": ws.cell(row, 9).value,
        "CV": ws.cell(row, 10).value,
        "CPI": ws.cell(row, 11).value,
        "SV_t": ws.cell(row, 12).value,
        "SPI_t": ws.cell(row, 13).value,
        "p_factor": ws.cell(row, 14).value,

        # Forecasting de plazo (EAC(t))
        "EAC_t_PV_PF1": ws.cell(row, 15).value,
        "EAC_t_ES_PF1": ws.cell(row, 21).value,
        "EAC_t_ES_SPIt": ws.cell(row, 22).value,

        # Forecasting de coste (EAC)
        "EAC_PF1": ws.cell(row, 24).value,
        "EAC_CPI": ws.cell(row, 25).value,
        "EAC_SPI": ws.cell(row, 26).value,
        "EAC_SPIt": ws.cell(row, 27).value,
        "EAC_080_CPI_020_SPI": ws.cell(row, 30).value,
        "EAC_080_CPI_020_SPIt": ws.cell(row, 31).value,
    })

df_tracking = pd.DataFrame(records)

# Convertir a numérico
numeric_cols = [
    "PV", "EV", "AC", "SV", "SPI", "CV", "CPI", "SPI_t", "p_factor",
    "EAC_t_PV_PF1", "EAC_t_ES_PF1", "EAC_t_ES_SPIt",
    "EAC_PF1", "EAC_CPI", "EAC_SPI", "EAC_SPIt",
    "EAC_080_CPI_020_SPI", "EAC_080_CPI_020_SPIt"
]

for col in numeric_cols:
    df_tracking[col] = pd.to_numeric(df_tracking[col], errors="coerce")

print(f"Project Tracking enriquecido: {len(df_tracking)} periodos")
print(f"Columnas: {list(df_tracking.columns)}")

## Variables de negocio customizadas.

Procedemos a calcular variables relevantes que enriquezcan el dataset

In [4]:
# BAC = PV del último periodo (cuando el proyecto está al 100%)
BAC = df_tracking["PV"].iloc[-1]
df_tracking["BAC"] = BAC

# Métricas adicionales calculadas
df_tracking["ETC_PF1"] = df_tracking["EAC_PF1"] - df_tracking["AC"]          # Estimate to Complete
df_tracking["VAC_PF1"] = BAC - df_tracking["EAC_PF1"]                        # Variance at Completion

df_tracking["ETC_CPI"] = df_tracking["EAC_CPI"] - df_tracking["AC"]
df_tracking["VAC_CPI"] = BAC - df_tracking["EAC_CPI"]

# % completado y % gastado
df_tracking["Pct_Complete"] = df_tracking["EV"] / BAC * 100
df_tracking["Pct_Spent"] = df_tracking["AC"] / BAC * 100

print(f"BAC del proyecto: {BAC:,.0f} €")
print()
print(df_tracking[["TP", "Status_Date", "PV", "EV", "AC", "SPI", "CPI", "EAC_PF1", "EAC_CPI"]].tail(6))

### Comprobación

In [5]:
print("=== Comprobación Forecasting ===")
last = df_tracking.iloc[-1]

print(f"Último periodo (TP {int(last['TP'])})")
print(f"Status Date     : {last['Status_Date']}")
print(f"PV / EV / AC    : {last['PV']:,.0f} / {last['EV']:,.0f} / {last['AC']:,.0f}")
print(f"SPI / CPI       : {last['SPI']:.3f} / {last['CPI']:.3f}")
print()
print("Estimaciones de coste final (EAC):")
print(f"  EAC (PF=1)    : {last['EAC_PF1']:,.0f} €")
print(f"  EAC (PF=CPI)  : {last['EAC_CPI']:,.0f} €")
print(f"  EAC (PF=SPI)  : {last['EAC_SPI']:,.0f} €")
print()
print(f"% Complete      : {last['Pct_Complete']:.1f}%")
print(f"% Spent         : {last['Pct_Spent']:.1f}%")

## Guardar el Project Tracking enriquecido

In [6]:
df_tracking.to_excel(OUTPUT_DIR / "Project_Tracking_Enriched.xlsx", index=False)
print("Guardado: 04_Outputs/Project_Tracking_Enriched.xlsx")
print(f"Columnas totales: {len(df_tracking.columns)}")

# FASE 2.2 - Lógica de alarmas

## Definicion de alarmas

In [7]:
# -------------------------------------------------
# Reglas de alarma (claras y defendibles)
# -------------------------------------------------

# Umbrales
SPI_WARNING = 0.95
SPI_CRITICAL = 0.90

CPI_WARNING = 0.90
CPI_CRITICAL = 0.85

def assign_alarms(row):
    alarms = []

    # Alarmas de plazo (SPI)
    if pd.notna(row["SPI"]):
        if row["SPI"] < SPI_CRITICAL:
            alarms.append("SPI_CRITICAL")
        elif row["SPI"] < SPI_WARNING:
            alarms.append("SPI_WARNING")

    # Alarmas de coste (CPI)
    if pd.notna(row["CPI"]):
        if row["CPI"] < CPI_CRITICAL:
            alarms.append("CPI_CRITICAL")
        elif row["CPI"] < CPI_WARNING:
            alarms.append("CPI_WARNING")

    # Alarma combinada
    if "SPI_CRITICAL" in alarms and "CPI_CRITICAL" in alarms:
        alarms.append("DOUBLE_CRITICAL")

    return alarms

df_tracking["Alarms"] = df_tracking.apply(assign_alarms, axis=1)
df_tracking["Has_Alarm"] = df_tracking["Alarms"].apply(lambda x: len(x) > 0)
df_tracking["Alarm_Level"] = df_tracking["Alarms"].apply(
    lambda x: "CRITICAL" if any("CRITICAL" in a for a in x) else ("WARNING" if x else "OK")
)

print("Reglas de alarma aplicadas")
print()
print(df_tracking["Alarm_Level"].value_counts())

## Primeras Alarmas

In [8]:
# Primeras alarmas de cada tipo
print("=== Primeras alarmas detectadas ===\n")

for alarm_type in ["SPI_WARNING", "SPI_CRITICAL", "CPI_WARNING", "CPI_CRITICAL", "DOUBLE_CRITICAL"]:
    mask = df_tracking["Alarms"].apply(lambda x: alarm_type in x)
    if mask.any():
        first = df_tracking[mask].iloc[0]
        print(f"{alarm_type:20} → TP {int(first['TP']):3d} | {first['Status_Date']} | SPI={first['SPI']:.3f} | CPI={first['CPI']:.3f}")
    else:
        print(f"{alarm_type:20} → No se ha disparado")

## Historico de alarmas a lo largo del proyecto

In [11]:
print("=== Evolución de alarmas ===\n")

summary = df_tracking.groupby("Alarm_Level").agg(
    Periodos=("TP", "count"),
    Primer_TP=("TP", "min"),
    Ultimo_TP=("TP", "max")
).reindex(["OK", "WARNING", "CRITICAL"])

print(summary)

In [12]:
# Creamos una versión más legible de las alarmas
df_alarms = df_tracking[[
    "TP", "Status_Date", "PV", "EV", "AC",
    "SPI", "CPI", "SPI_t",
    "EAC_PF1", "EAC_CPI",
    "Alarm_Level", "Alarms"
]].copy()

df_alarms.to_excel(OUTPUT_DIR / "Project_Tracking_with_Alarms.xlsx", index=False)
print("Guardado: 04_Outputs/Project_Tracking_with_Alarms.xlsx")

## Resumen Alarmas

In [13]:
print("=" * 65)
print("RESUMEN DE CONTROL DEL PROYECTO")
print("=" * 65)
print(f"Periodos totales          : {len(df_tracking)}")
print(f"Periodos OK               : {(df_tracking['Alarm_Level']=='OK').sum()}")
print(f"Periodos CRITICAL         : {(df_tracking['Alarm_Level']=='CRITICAL').sum()}")
print()
print("Momentos clave:")
print(f"  Primera alarma de plazo : TP 23 (SPI Critical)")
print(f"  Primera alarma de coste : TP 72 (CPI Critical)")
print(f"  Inicio Double Critical  : TP 72")
print()
print("Estado final:")
print(f"  SPI final               : {df_tracking.iloc[-1]['SPI']:.3f}")
print(f"  CPI final               : {df_tracking.iloc[-1]['CPI']:.3f}")
print(f"  Sobrecoste final        : {df_tracking.iloc[-1]['Pct_Spent'] - 100:.1f}%")
print("=" * 65)

# FASE 2.3 - Analisis de deviaciones

## Preparación de datos para el análisis.

In [20]:
# Aseguramos tipos y orden
df = df_tracking.copy()
df = df.sort_values("TP").reset_index(drop=True)

# Variables de apoyo
df["SPI_deviation"] = 1 - df["SPI"]
df["CPI_deviation"] = 1 - df["CPI"]
df["Cost_Overrun_Pct"] = (df["AC"] / df["BAC"] - 1) * 100
df["Schedule_Overrun_Flag"] = df["SPI"] < 1
df["Cost_Overrun_Flag"] = df["CPI"] < 1

print("Datos preparados para análisis de desviaciones")
print(f"Periodos analizados: {len(df)}")

## Resumen estadístico de desviaciones

In [22]:
print("=" * 65)
print("RESUMEN ESTADÍSTICO DE DESVIACIONES")
print("=" * 65)

print("\n--- SPI (Plazo) ---")
print(f"SPI medio              : {df['SPI'].mean():.3f}")
print(f"SPI mínimo             : {df['SPI'].min():.3f}")
print(f"SPI máximo             : {df['SPI'].max():.3f}")
print(f"Periodos con SPI < 1.0 : {(df['SPI'] < 1).sum()} de {len(df)}")
print(f"Periodos con SPI < 0.9 : {(df['SPI'] < 0.9).sum()} de {len(df)}")

print("\n--- CPI (Coste) ---")
print(f"CPI medio              : {df['CPI'].mean():.3f}")
print(f"CPI mínimo             : {df['CPI'].min():.3f}")
print(f"CPI máximo             : {df['CPI'].max():.3f}")
print(f"Periodos con CPI < 1.0 : {(df['CPI'] < 1).sum()} de {len(df)}")
print(f"Periodos con CPI < 0.85: {(df['CPI'] < 0.85).sum()} de {len(df)}")

print("\n--- Estado final ---")
print(f"SPI final              : {df.iloc[-1]['SPI']:.3f}")
print(f"CPI final              : {df.iloc[-1]['CPI']:.3f}")
print(f"Sobrecoste final       : {df.iloc[-1]['Cost_Overrun_Pct']:.1f}%")

## Puntos de inflexion

In [24]:
print("=" * 65)
print("MOMENTOS CLAVE DE INFLEXIÓN")
print("=" * 65)

# Primera vez que SPI baja de 0.95
first_spi_warn = df[df["SPI"] < 0.95].iloc[0]
print(f"\n1. Primera señal de problema de plazo (SPI < 0.95)")
print(f"   TP {int(first_spi_warn['TP'])} | {first_spi_warn['Status_Date']}")
print(f"   SPI = {first_spi_warn['SPI']:.3f} | CPI = {first_spi_warn['CPI']:.3f}")

# Primera vez que SPI baja de 0.90
first_spi_crit = df[df["SPI"] < 0.90].iloc[0]
print(f"\n2. Entrada en zona crítica de plazo (SPI < 0.90)")
print(f"   TP {int(first_spi_crit['TP'])} | {first_spi_crit['Status_Date']}")
print(f"   SPI = {first_spi_crit['SPI']:.3f} | CPI = {first_spi_crit['CPI']:.3f}")

# Primera vez que CPI baja de 0.90
first_cpi_warn = df[df["CPI"] < 0.90]
if len(first_cpi_warn) > 0:
    first_cpi_warn = first_cpi_warn.iloc[0]
    print(f"\n3. Primera señal de problema de coste (CPI < 0.90)")
    print(f"   TP {int(first_cpi_warn['TP'])} | {first_cpi_warn['Status_Date']}")
    print(f"   SPI = {first_cpi_warn['SPI']:.3f} | CPI = {first_cpi_warn['CPI']:.3f}")

# Primera vez que CPI baja de 0.85
first_cpi_crit = df[df["CPI"] < 0.85]
if len(first_cpi_crit) > 0:
    first_cpi_crit = first_cpi_crit.iloc[0]
    print(f"\n4. Entrada en zona crítica de coste (CPI < 0.85)")
    print(f"   TP {int(first_cpi_crit['TP'])} | {first_cpi_crit['Status_Date']}")
    print(f"   SPI = {first_cpi_crit['SPI']:.3f} | CPI = {first_cpi_crit['CPI']:.3f}")

# Peor momento de SPI
worst_spi = df.loc[df["SPI"].idxmin()]
print(f"\n5. Peor momento de plazo")
print(f"   TP {int(worst_spi['TP'])} | {worst_spi['Status_Date']}")
print(f"   SPI = {worst_spi['SPI']:.3f} | CPI = {worst_spi['CPI']:.3f}")

# Peor momento de CPI
worst_cpi = df.loc[df["CPI"].idxmin()]
print(f"\n6. Peor momento de coste")
print(f"   TP {int(worst_cpi['TP'])} | {worst_cpi['Status_Date']}")
print(f"   SPI = {worst_cpi['SPI']:.3f} | CPI = {worst_cpi['CPI']:.3f}")

In [25]:
# ============================================================
# CIERRE FASE 2.3 – Conclusiones de negocio orientadas a MEINS
# ============================================================

print("=" * 70)
print("CONCLUSIONES DE CONTROL – PROYECTO WIND FARM C2011-13")
print("=" * 70)

print("""
1. COMPORTAMIENTO DEL PROYECTO
   - El plazo se deterioró muy rápido (desde TP 23) y llegó a un mínimo crítico
     (SPI = 0.201 en TP 66). Posteriormente se recuperó hasta SPI = 1.0 al final.
   - El coste se mantuvo relativamente controlado hasta la mitad del proyecto.
     A partir de TP 72 el CPI se desploma y el proyecto termina con un sobrecoste
     del 22 % (CPI final = 0.819).
   - Interpretación clásica de parques eólicos: se sacrificó coste para recuperar
     plazo (horas extra, proveedores premium, cambios de secuencia, etc.).

2. SEÑALES TEMPRANAS QUE UN PROJECT PLANNER / DATA ANALYST DEBE DETECTAR
   - TP 23: primera alarma fuerte de plazo (SPI < 0.70). Momento ideal para
     activar plan de recuperación y revisión de ruta crítica + materiales críticos.
   - TP 66: peor momento de plazo. Aquí ya se debería haber escalado a dirección
     y activado medidas extraordinarias.
   - TP 72: aparición simultánea de problemas de coste. Indica que las medidas
     de recuperación de plazo empezaron a generar sobrecoste estructural.

3. IMPLICACIONES PARA UN ROL COMO EL DE MEINS
   - Un sistema de alarmas como el que hemos construido hubiera permitido
     intervenir 40-50 periodos antes del peor momento de coste.
   - El tracking de 120 periodos demuestra la necesidad de dashboards diarios /
     semanales, no solo informes mensuales.
   - La recuperación de plazo a costa de coste es un patrón típico en fabricación
     y montaje de equipos renovables (torres, nacelles, blades). Hay que vigilar
     especialmente el CPI cuando se activa un plan de recuperación de SPI.

4. PRÓXIMOS PASOS LÓGICOS (alineados con la oferta MEINS)
   - Fase 3: Análisis de desviaciones por paquete de trabajo / fase del proyecto
     (permisos, civil, suministro de componentes críticos, montaje, comisionado).
   - Fase 4: Material Control & Logistics – identificar actividades de suministro
     crítico y simular un sistema de “material readiness” ligado al schedule.
   - Fase 5: Acciones correctivas proactivas + lecciones aprendidas + cómo
     escalar esto a un portfolio de fabricación de equipos industriales.
""")

print("=" * 70)
print("FASE 2 CERRADA CORRECTAMENTE")
print("=" * 70)

In [26]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Aseguramos que Status_Date es datetime
df["Status_Date"] = pd.to_datetime(df["Status_Date"])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Wind Farm C2011-13 – Control de Proyecto (EVM + Alarmas)", fontsize=14, fontweight="bold")

# 1. Evolución SPI vs CPI
ax1 = axes[0, 0]
ax1.plot(df["Status_Date"], df["SPI"], label="SPI", linewidth=2)
ax1.plot(df["Status_Date"], df["CPI"], label="CPI", linewidth=2)
ax1.axhline(1.0, color="gray", linestyle="--", alpha=0.7)
ax1.axhline(0.90, color="orange", linestyle="--", alpha=0.6, label="Warning")
ax1.axhline(0.85, color="red", linestyle="--", alpha=0.6, label="Critical (CPI)")
ax1.set_title("Evolución SPI vs CPI")
ax1.set_ylabel("Índice")
ax1.legend(loc="lower left")
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# 2. PV / EV / AC
ax2 = axes[0, 1]
ax2.plot(df["Status_Date"], df["PV"]/1e6, label="PV", linewidth=2)
ax2.plot(df["Status_Date"], df["EV"]/1e6, label="EV", linewidth=2)
ax2.plot(df["Status_Date"], df["AC"]/1e6, label="AC", linewidth=2)
ax2.set_title("Curvas S – PV / EV / AC (M€)")
ax2.set_ylabel("Millones €")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# 3. Alarm Level a lo largo del tiempo
ax3 = axes[1, 0]
alarm_map = {"OK": 0, "WARNING": 1, "CRITICAL": 2}
df["Alarm_Num"] = df["Alarm_Level"].map(alarm_map)
ax3.fill_between(df["Status_Date"], df["Alarm_Num"], step="mid", alpha=0.7)
ax3.set_yticks([0, 1, 2])
ax3.set_yticklabels(["OK", "WARNING", "CRITICAL"])
ax3.set_title("Nivel de Alarma a lo largo del proyecto")
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# 4. Forecast de coste (EAC)
ax4 = axes[1, 1]
if "EAC_CPI" in df.columns:
    ax4.plot(df["Status_Date"], df["EAC_CPI"]/1e6, label="EAC (PF=CPI)", linewidth=2)
if "EAC_PF1" in df.columns:
    ax4.plot(df["Status_Date"], df["EAC_PF1"]/1e6, label="EAC (PF=1)", linewidth=1.5, alpha=0.8)
ax4.axhline(df["BAC"].iloc[0]/1e6, color="green", linestyle="--", label="BAC")
ax4.set_title("Forecast de Coste Final (EAC)")
ax4.set_ylabel("Millones €")
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_Control_Dashboard_Overview.png", dpi=150, bbox_inches="tight")
plt.show()

print("Gráfico guardado en: 04_Outputs/04_Control_Dashboard_Overview.png")

### Estado del Notebook 04
- Enriquecimiento con forecasting → Completado
- Sistema de alarmas → Completado
- Análisis de desviaciones e inflexiones → Completado
- Conclusiones de negocio orientadas a MEINS → Completado
- Visualizaciones clave de control → Completado

**Siguiente paso:** Notebook 05 – Material Control & Logistics (versión enfocada y realista)